# 05C — Evaluate Checkpoints + Classical Baselines

Produces all report-ready numbers without retraining. No GPU needed (but faster with one).

### Datasets to attach
| Dataset | Role |
|---|---|
| Your code zip (`brats-seg-code`) | Source code |
| `brats2020-2d-prepared` (05A output) | Test slices |
| Each 05B training output | Checkpoints (`.pth` files) |

Attach as many checkpoint datasets as you have. Missing ones are skipped automatically.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
CODE_DATASET_SLUG = 'brats-seg-code'

# 200-300 for a draft. None for final report (slow — runs all test slices).
CLASSICAL_LIMIT = 300

# Maps experiment name → config file. All five listed; missing checkpoints are skipped.
EXPERIMENT_CONFIGS = {
    'brats_baseline':            'configs/default.yaml',
    'unet_best_fusion_loss_aug': 'configs/unet_best_fusion_loss_aug.yaml',
    'unetpp':                    'configs/unetpp.yaml',
    'attention_unet':            'configs/attention_unet.yaml',
    'uncertainty_unet':          'configs/uncertainty_unet.yaml',
}

In [ ]:
import os, shutil, subprocess, sys, json, yaml
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

def disk_gb(path='/'):
    st = os.statvfs(path)
    return round(st.f_bavail * st.f_frsize / 1e9, 2)

print(f'Free disk: {disk_gb()} GB')

In [ ]:
# ── 1. Copy repo ───────────────────────────────────────────────────────────
def looks_like_repo(p):
    return (p / 'src').exists() and (p / 'scripts').exists() and (p / 'configs').exists()

repo_input = Path('/kaggle/input') / CODE_DATASET_SLUG
if not looks_like_repo(repo_input):
    for p in Path('/kaggle/input').rglob('requirements.txt'):
        if looks_like_repo(p.parent):
            repo_input = p.parent; break

PROJECT = Path('/kaggle/working/project')
if PROJECT.exists(): shutil.rmtree(PROJECT)
shutil.copytree(repo_input, PROJECT,
    ignore=shutil.ignore_patterns('.git', '__pycache__', '.pytest_cache', 'data', 'outputs', 'checkpoints'))
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

def run(cmd):
    subprocess.check_call(cmd, cwd=str(PROJECT))

print('Repo:', repo_input)

In [ ]:
# ── 2. Install missing packages ────────────────────────────────────────────
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nibabel', 'albumentations'])
print('Packages ready.')

In [ ]:
# ── 3. Find prepared data ──────────────────────────────────────────────────
PREPARED = next(
    (p for p in [
        Path('/kaggle/input/brats2020-2d-prepared'),
        Path('/kaggle/input/brats2020_2d_prepared'),
    ] if (p / 'data/brats2020_2d/metadata.json').exists()),
    None
)
if PREPARED is None:
    raise FileNotFoundError('Attach the brats2020-2d-prepared dataset (output of 05A).')

DATA_ROOT  = PREPARED / 'data/brats2020_2d'
SPLIT_FILE = PREPARED / 'configs/splits/default.json'
print(f'Prepared data: {DATA_ROOT}')

In [ ]:
# ── 4. Build runtime configs (patch absolute paths) ────────────────────────
RT_DIR = Path('/kaggle/working/rt_configs')
RT_DIR.mkdir(parents=True, exist_ok=True)

def make_rt_config(cfg_path: str) -> Path:
    with open(PROJECT / cfg_path) as f:
        cfg = yaml.safe_load(f)
    cfg['data']['data_root']  = str(DATA_ROOT)
    cfg['data']['split_file'] = str(SPLIT_FILE)
    cfg['checkpoint_dir']     = '/kaggle/working/checkpoints'
    cfg['experiment']['output_dir'] = '/kaggle/working/outputs'
    out = RT_DIR / Path(cfg_path).name
    with open(out, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return out

def find_checkpoint(exp_name: str) -> Path | None:
    # Search both /kaggle/input (attached outputs) and /kaggle/working
    pattern = f'{exp_name}_best.pth'
    hits = list(Path('/kaggle/input').rglob(pattern)) + list(Path('/kaggle/working').rglob(pattern))
    return hits[0] if hits else None

print('Checkpoint discovery:')
for exp in EXPERIMENT_CONFIGS:
    ckpt = find_checkpoint(exp)
    status = str(ckpt) if ckpt else 'NOT FOUND (will skip)'
    print(f'  {exp:<35} {status}')

In [ ]:
# ── 5. Classical baselines ─────────────────────────────────────────────────
rt_default = make_rt_config('configs/default.yaml')
cmd = [sys.executable, '-m', 'scripts.classical_baselines',
       '--config', str(rt_default), '--split', 'test']
if CLASSICAL_LIMIT:
    cmd += ['--limit', str(CLASSICAL_LIMIT)]
print('Running classical baselines …')
run(cmd)
print('Done.')

In [ ]:
# ── 6. Evaluate each checkpoint ────────────────────────────────────────────
for exp, cfg_path in EXPERIMENT_CONFIGS.items():
    ckpt = find_checkpoint(exp)
    if ckpt is None:
        print(f'[SKIP] {exp} — checkpoint not attached')
        continue
    rt = make_rt_config(cfg_path)
    print(f'\n=== Evaluating {exp} ===')
    run([
        sys.executable, '-m', 'scripts.evaluate',
        '--config',     str(rt),
        '--checkpoint', str(ckpt),
        '--split',      'test',
    ])

In [ ]:
# ── 7. Collect and display results table ───────────────────────────────────
import pandas as pd

rows = []
outputs_dir = Path('/kaggle/working/outputs')

# Classical baselines
classical_csv = outputs_dir / 'brats_baseline/classical_baselines.csv'
if classical_csv.exists():
    df_c = pd.read_csv(classical_csv)
    for method in df_c['method'].unique():
        sub = df_c[df_c['method'] == method]
        rows.append({
            'model': f'Classical: {method}',
            'mean_dice': sub['mean_dice'].mean(),
            'dice_NCR/NET': sub.get('dice_NCR/NET', pd.Series([float('nan')])).mean(),
            'dice_edema': sub.get('dice_edema', pd.Series([float('nan')])).mean(),
            'dice_enhancing_tumor': sub.get('dice_enhancing_tumor', pd.Series([float('nan')])).mean(),
        })

# Deep learning models
for exp in EXPERIMENT_CONFIGS:
    metrics_json = outputs_dir / exp / 'test_metrics.json'
    if metrics_json.exists():
        with open(metrics_json) as f:
            m = json.load(f)
        rows.append({
            'model': exp,
            'mean_dice': m.get('mean_dice', float('nan')),
            'dice_NCR/NET': m.get('dice_NCR/NET', float('nan')),
            'dice_edema': m.get('dice_edema', float('nan')),
            'dice_enhancing_tumor': m.get('dice_enhancing_tumor', float('nan')),
        })

if rows:
    df = pd.DataFrame(rows).set_index('model')
    print(df.round(4).to_string())

    # Bar chart
    df['mean_dice'].plot(kind='barh', figsize=(10, 6), color='steelblue', edgecolor='white')
    plt.xlabel('Mean Dice (excl. background)')
    plt.title('Model Comparison — Test Set Dice')
    plt.tight_layout()
    plt.show()
else:
    print('No results yet — run cells 5 and 6 first.')

In [ ]:
# ── 8. Package all evaluation outputs ─────────────────────────────────────
EXPORT = Path('/kaggle/working/evaluation_artifacts')
if EXPORT.exists(): shutil.rmtree(EXPORT)
EXPORT.mkdir()
if outputs_dir.exists():
    shutil.copytree(outputs_dir, EXPORT / 'outputs')
archive = shutil.make_archive(str(EXPORT), 'zip', EXPORT)
print('Packaged:', archive)
print(f'Free disk: {disk_gb()} GB')